In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 🔧 Preprocessing & Feature Selection
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, 
                           roc_auc_score, precision_recall_curve, average_precision_score,
                           f1_score)

# 🎯 XGBoost and sampling
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# 📥 Load and preprocess data
print("📥 Loading data...")
data = pd.read_csv('Transactions.csv')

# 🕒 Datetime feature engineering
data['trans_date_trans_time'] = pd.to_datetime(data['trans_date_trans_time'])

# Sort by datetime to ensure monotonicity
data = data.sort_values('trans_date_trans_time').reset_index(drop=True)

data['trans_hour'] = data['trans_date_trans_time'].dt.hour
data['trans_day'] = data['trans_date_trans_time'].dt.dayofweek
data['trans_month'] = data['trans_date_trans_time'].dt.month
data['trans_year'] = data['trans_date_trans_time'].dt.year
data['trans_date'] = data['trans_date_trans_time'].dt.to_period('M').dt.to_timestamp()

# 🎂 Age calculation
data['dob'] = pd.to_datetime(data['dob'])
data['age'] = np.round((data['trans_date'] - data['dob']).dt.days / 365.25)

# ⏰ Temporal features
data['is_weekend'] = data['trans_day'].isin([5, 6])
data['is_night'] = data['trans_hour'].between(20, 23) | data['trans_hour'].between(0, 4)
data['is_morning_rush'] = data['trans_hour'].between(7, 9)
data['is_lunch_time'] = data['trans_hour'].between(11, 13)
data['afternoon'] = data['trans_hour'].between(12, 17)
data['late_night'] = data['trans_hour'].between(0, 5)

# 💳 Transaction frequency features
print("💳 Creating transaction frequency features...")

# Time since last transaction
data = data.sort_values(['cc_num', 'trans_date_trans_time'])
data['time_since_last_tx'] = data.groupby('cc_num')['trans_date_trans_time'].diff().dt.total_seconds() / 3600
data['time_since_last_tx'] = data['time_since_last_tx'].fillna(24)

# Daily transaction pattern
data['day_of_month'] = data['trans_date_trans_time'].dt.day
data['is_month_end'] = data['day_of_month'] >= 25

# Amount statistics per card
card_stats = data.groupby('cc_num')['amt'].agg(['mean', 'std']).reset_index()
card_stats.columns = ['cc_num', 'card_amt_mean', 'card_amt_std']
data = data.merge(card_stats, on='cc_num', how='left')

data['amt_to_card_mean_ratio'] = data['amt'] / data['card_amt_mean']
data['amt_z_score'] = (data['amt'] - data['card_amt_mean']) / data['card_amt_std']
data['amt_z_score'] = data['amt_z_score'].fillna(0)

# 🎯 Amount-based features
data['amt_log'] = np.log1p(data['amt'])
data['is_high_amount'] = data['amt'] > data['amt'].quantile(0.95)
data['is_low_amount'] = data['amt'] < data['amt'].quantile(0.05)

# 🗺️ Distance features
data['dist_merch2cust'] = np.sqrt(
    (data['lat'] - data['merch_lat'])**2 + 
    (data['long'] - data['merch_long'])**2
)
data['is_long_distance'] = data['dist_merch2cust'] > data['dist_merch2cust'].quantile(0.95)

# 🗑️ Drop unnecessary columns
cols_to_drop = ['trans_num', 'trans_date_trans_time', 'trans_date', 'dob', 'day_of_month', 
                'card_amt_mean', 'card_amt_std']
cols_to_drop = [col for col in cols_to_drop if col in data.columns]
data.drop(cols_to_drop, axis=1, inplace=True)

target = 'is_fraud'
X = data.drop(columns=[target])
y = data[target]

print(f"📊 Dataset shape: {X.shape}")
print(f"🎯 Fraud rate: {y.mean():.4f} ({y.sum()} fraud cases)")

# 🧼 Encode categorical features
print("🔤 Encoding categorical features...")
cat_cols = X.select_dtypes(include='object').columns
for col in cat_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# 🧼 Encode boolean features
bool_cols = X.select_dtypes(include='bool').columns
for col in bool_cols:
    X[col] = X[col].astype(int)

print(f"✅ Final feature set: {X.shape[1]} features")

# 📊 Train-test-holdout split
print("📊 Creating train-test-holdout split...")
X_temp, X_hold, y_temp, y_hold = train_test_split(
    X, y, test_size=0.003, stratify=y, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42
)

print(f"Train set: {X_train.shape}, Fraud rate: {y_train.mean():.4f}")
print(f"Test set: {X_test.shape}, Fraud rate: {y_test.mean():.4f}")
print(f"Holdout set: {X_hold.shape}, Fraud rate: {y_hold.mean():.4f}")

# 🔍 Feature Selection for XGBoost
print("\n🔍 Performing feature selection...")

# Variance Threshold
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_train)
selected_vt_features = X_train.columns[vt.get_support()].tolist()

if len(selected_vt_features) == 0:
    print("⚠️  Variance Threshold removed all features! Using all features instead.")
    selected_vt_features = X_train.columns.tolist()

print(f"📈 Variance Threshold selected {len(selected_vt_features)} features")

# Mutual Information for feature ranking
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train[selected_vt_features])
mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=42)

mi_df = pd.DataFrame({
    'Feature': selected_vt_features, 
    'Mutual_Info': mi_scores
}).sort_values(by='Mutual_Info', ascending=False)

# XGBoost Feature Importance
xgb_base = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_base.fit(X_train[selected_vt_features], y_train)
xgb_importance = xgb_base.feature_importances_

xgb_df = pd.DataFrame({
    'Feature': selected_vt_features, 
    'XGB_Importance': xgb_importance
}).sort_values(by='XGB_Importance', ascending=False)

# 📊 Combine scores
score_df = mi_df.merge(xgb_df, on='Feature')
score_df['Composite_Score'] = score_df[['Mutual_Info', 'XGB_Importance']].mean(axis=1)
score_df = score_df.sort_values(by='Composite_Score', ascending=False)

print("\n🏆 Top 20 Features for XGBoost:")
print(score_df.head(20)[['Feature', 'Composite_Score']])

# ✅ Final feature selection
top_n = min(30, len(score_df))  # Select top 30 features
top_features = score_df.head(top_n)['Feature'].tolist()
print(f"🎯 Selected {len(top_features)} top features for XGBoost")

# Apply feature selection
X_train_xgb = X_train[top_features]
X_test_xgb = X_test[top_features]
X_hold_xgb = X_hold[top_features]

# 🎯 Handle Class Imbalance with SMOTE
print("\n🔄 Applying SMOTE for class imbalance...")
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train_xgb, y_train)

print(f"Original train: {X_train_xgb.shape}, Fraud: {y_train.sum()}")
print(f"After SMOTE: {X_smote.shape}, Fraud: {y_smote.sum()}")

📥 Loading data...
💳 Creating transaction frequency features...
📊 Dataset shape: (1296675, 41)
🎯 Fraud rate: 0.0058 (7506 fraud cases)
🔤 Encoding categorical features...
✅ Final feature set: 41 features
📊 Creating train-test-holdout split...
Train set: (1034227, 41), Fraud rate: 0.0058
Test set: (258557, 41), Fraud rate: 0.0058
Holdout set: (3891, 41), Fraud rate: 0.0059

🔍 Performing feature selection...
📈 Variance Threshold selected 37 features

🏆 Top 20 Features for XGBoost:
                   Feature  Composite_Score
0                 is_night         0.217859
11                     amt         0.170737
5   is_high_risk_merch_cat         0.042667
9                 category         0.038359
1               is_weekend         0.037646
2                afternoon         0.035582
10              trans_hour         0.031016
3               trans_year         0.027845
4                trans_day         0.021161
6             is_month_end         0.015764
16             amt_z_score        

In [2]:
# 🧠 XGBoost Hyperparameter Tuning
print("\n🧠 Training XGBoost with optimized parameters...")

# Define XGBoost parameters for fraud detection
xgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 8,
    'min_child_weight': 1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'eval_metric': 'aucpr',
    'use_label_encoder': False
}

# 📊 Enhanced Evaluation Function for XGBoost
def evaluate_xgb_model(clf, X_train, y_train, X_test, y_test, model_name, use_smote=False):
    """Evaluate XGBoost model with comprehensive metrics"""
    
    # Train on SMOTE data if specified
    if use_smote:
        X_train_use, y_train_use = X_smote, y_smote
    else:
        X_train_use, y_train_use = X_train, y_train
    
    # Train model with early stopping
    clf.fit(
        X_train_use, y_train_use,
        eval_set=[(X_test, y_test)],
        early_stopping_rounds=50,
        verbose=False
    )
    
    # Predictions
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    
    # Comprehensive metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_proba)
    avg_precision = average_precision_score(y_test, y_proba)
    
    # Fraud class metrics
    fraud_report = report.get('1', {'precision': 0, 'recall': 0, 'f1-score': 0})
    fraud_f1 = f1_score(y_test, y_pred, pos_label=1)
    
    # Geometric Mean
    g_mean = np.sqrt(fraud_report['recall'] * report['0']['recall'])
    
    return {
        'Model': model_name,
        'ROC_AUC': roc_auc,
        'Average_Precision': avg_precision,
        'Fraud_Precision': fraud_report['precision'],
        'Fraud_Recall': fraud_report['recall'],
        'Fraud_F1': fraud_f1,
        'G_Mean': g_mean,
        'Accuracy': accuracy_score(y_test, y_pred),
        'y_proba': y_proba,
        'Model_Object': clf,
        'Feature_Importance': clf.feature_importances_,
        'Best_Iteration': clf.best_iteration if hasattr(clf, 'best_iteration') else xgb_params['n_estimators']
    }

# 🎯 Find Optimal Threshold for Fraud Detection
def find_optimal_threshold(y_true, y_proba):
    """Find optimal threshold based on F1 score for fraud class"""
    thresholds = np.arange(0.05, 0.95, 0.02)
    best_threshold = 0.5
    best_f1 = 0
    
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, pos_label=1)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    return best_threshold



🧠 Training XGBoost with optimized parameters...


In [3]:

# 📈 Train and Evaluate XGBoost Variants
print("\n📈 Training XGBoost variants...")

xgb_results = []

# Train different XGBoost configurations
xgb_configs = [
    {
        'name': 'XGBoost Scaled Weight',
        'params': {**xgb_params, **{'scale_pos_weight': len(y_train[y_train==0]) / len(y_train[y_train==1])}},
        'use_smote': False
    },
    {
        'name': 'XGBoost SMOTE', 
        'params': {**xgb_params, **{'scale_pos_weight': len(y_train[y_train==0]) / len(y_train[y_train==1])}},
        'use_smote': True
    },
    {
        'name': 'XGBoost Balanced',
        'params': {**xgb_params, **{'scale_pos_weight': 1}},  # Balanced approach
        'use_smote': False
    },
    {
        'name': 'XGBoost High Weight',
        'params': {**xgb_params, **{'scale_pos_weight': 50}},  # Higher weight for fraud class
        'use_smote': False
    }
]

for config in xgb_configs:
    try:
        print(f"Training {config['name']}...")
        clf = XGBClassifier(**config['params'])
        
        result = evaluate_xgb_model(
            clf, X_train_xgb, y_train, X_test_xgb, y_test, 
            config['name'], use_smote=config['use_smote']
        )
        xgb_results.append(result)
        
        print(f"✅ {config['name']}: "
              f"AP={result['Average_Precision']:.4f}, "
              f"Fraud F1={result['Fraud_F1']:.4f}, "
              f"Best Iteration={result['Best_Iteration']}")
              
    except Exception as e:
        print(f"❌ {config['name']} failed: {e}")
        # Create a simple model without early stopping as fallback
        try:
            print(f"Trying fallback for {config['name']}...")
            simple_params = config['params'].copy()
            simple_params['n_estimators'] = 100  # Use fewer estimators
            clf_simple = XGBClassifier(**simple_params)
            clf_simple.fit(X_train_xgb, y_train)
            
            y_pred = clf_simple.predict(X_test_xgb)
            y_proba = clf_simple.predict_proba(X_test_xgb)[:, 1]
            
            report = classification_report(y_test, y_pred, output_dict=True)
            fraud_report = report.get('1', {'precision': 0, 'recall': 0, 'f1-score': 0})
            
            fallback_result = {
                'Model': config['name'] + ' (Fallback)',
                'ROC_AUC': roc_auc_score(y_test, y_proba),
                'Average_Precision': average_precision_score(y_test, y_proba),
                'Fraud_Precision': fraud_report['precision'],
                'Fraud_Recall': fraud_report['recall'], 
                'Fraud_F1': f1_score(y_test, y_pred, pos_label=1),
                'G_Mean': np.sqrt(fraud_report['recall'] * report['0']['recall']),
                'Accuracy': accuracy_score(y_test, y_pred),
                'y_proba': y_proba,
                'Model_Object': clf_simple,
                'Feature_Importance': clf_simple.feature_importances_,
                'Best_Iteration': 100
            }
            xgb_results.append(fallback_result)
            print(f"✅ {config['name']} (Fallback): AP={fallback_result['Average_Precision']:.4f}")
            
        except Exception as e2:
            print(f"❌ Fallback also failed for {config['name']}: {e2}")

# Check if we have any results
if not xgb_results:
    print("❌ All models failed! Training a simple default model...")
    # Train a simple default model
    default_model = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
    default_model.fit(X_train_xgb, y_train)
    
    y_pred = default_model.predict(X_test_xgb)
    y_proba = default_model.predict_proba(X_test_xgb)[:, 1]
    
    report = classification_report(y_test, y_pred, output_dict=True)
    fraud_report = report.get('1', {'precision': 0, 'recall': 0, 'f1-score': 0})
    
    default_result = {
        'Model': 'XGBoost Default',
        'ROC_AUC': roc_auc_score(y_test, y_proba),
        'Average_Precision': average_precision_score(y_test, y_proba),
        'Fraud_Precision': fraud_report['precision'],
        'Fraud_Recall': fraud_report['recall'],
        'Fraud_F1': f1_score(y_test, y_pred, pos_label=1),
        'G_Mean': np.sqrt(fraud_report['recall'] * report['0']['recall']),
        'Accuracy': accuracy_score(y_test, y_pred),
        'y_proba': y_proba,
        'Model_Object': default_model,
        'Feature_Importance': default_model.feature_importances_,
        'Best_Iteration': 100
    }
    xgb_results.append(default_result)

# 📊 Select Best XGBoost Model
xgb_results_df = pd.DataFrame(xgb_results)

print("\n" + "="*80)
print("🏆 XGBOOST MODEL COMPARISON")
print("="*80)

if not xgb_results_df.empty:
    xgb_results_df = xgb_results_df.sort_values('Average_Precision', ascending=False)
    print(xgb_results_df[['Model', 'Average_Precision', 'Fraud_F1', 'Fraud_Recall', 'ROC_AUC']].round(4))

    # 🏆 Select Best Model
    best_xgb_result = xgb_results_df.iloc[0]
    best_xgb_model = best_xgb_result['Model_Object']
    best_xgb_name = best_xgb_result['Model']

    print(f"\n🏆 BEST XGBOOST MODEL: {best_xgb_name}")

    # 🎯 Threshold Optimization
    y_test_proba = best_xgb_result['y_proba']
    optimal_threshold = find_optimal_threshold(y_test, y_test_proba)
    print(f"🎯 Optimal threshold: {optimal_threshold:.3f}")

    # 📈 Final Evaluation with Optimal Threshold
    y_test_pred_optimal = (y_test_proba >= optimal_threshold).astype(int)

    print("\n" + "="*80)
    print("🎯 FINAL XGBOOST PERFORMANCE (Test Set)")
    print("="*80)
    print(f"Model: {best_xgb_name}")
    print(f"Optimal Threshold: {optimal_threshold:.3f}")

    print("\n📊 Classification Report with Optimal Threshold:")
    print(classification_report(y_test, y_test_pred_optimal))

    conf_matrix = confusion_matrix(y_test, y_test_pred_optimal)
    print("📊 Confusion Matrix:")
    print(conf_matrix)

    print(f"📈 ROC AUC: {roc_auc_score(y_test, y_test_proba):.4f}")
    print(f"📊 Average Precision: {average_precision_score(y_test, y_test_proba):.4f}")

    # 🔧 Retrain Best Model on Full Training Data
    print(f"\n🔧 Retraining best XGBoost model on full training data...")

    # Determine if we should use SMOTE based on best model
    use_smote_final = 'SMOTE' in best_xgb_name

    if use_smote_final:
        X_final_train, y_final_train = smote.fit_resample(X_temp[top_features], y_temp)
    else:
        X_final_train, y_final_train = X_temp[top_features], y_temp

    # Use the best iteration from early stopping
    final_params = best_xgb_model.get_params()
    final_params['n_estimators'] = best_xgb_result['Best_Iteration']
    
    final_model = XGBClassifier(**final_params)
    final_model.fit(X_final_train, y_final_train)

    print(f"✅ Final model trained on {len(X_final_train)} samples with {final_params['n_estimators']} estimators")

    # 🎯 Holdout Set Evaluation
    print("\n" + "="*80)
    print("🔒 HOLDOUT SET EVALUATION")
    print("="*80)

    y_hold_proba = final_model.predict_proba(X_hold_xgb)[:, 1]
    y_hold_pred = (y_hold_proba >= optimal_threshold).astype(int)

    print("📊 Classification Report (Holdout Set):")
    print(classification_report(y_hold, y_hold_pred))

    holdout_conf_matrix = confusion_matrix(y_hold, y_hold_pred)
    print("📊 Confusion Matrix (Holdout):")
    print(holdout_conf_matrix)

    holdout_ap = average_precision_score(y_hold, y_hold_proba)
    holdout_auc = roc_auc_score(y_hold, y_hold_proba)
    print(f"📈 Holdout Average Precision: {holdout_ap:.4f}")
    print(f"📊 Holdout ROC AUC: {holdout_auc:.4f}")

    # 💾 Save XGBoost Model and Results
    print("\n💾 Saving XGBoost model and results...")

    import joblib

    # Save final model
    joblib.dump(final_model, 'xgb_fraud_model.pkl')
    joblib.dump(top_features, 'xgb_selected_features.pkl')

    # Save datasets
    holdout_set = pd.concat([X_hold_xgb, y_hold], axis=1)
    holdout_set.to_csv('xgb_holdout_set.csv', index=False)

    training_set = pd.concat([X_temp[top_features], y_temp], axis=1)
    training_set.to_csv('xgb_training_set.csv', index=False)

    # Save predictions
    predictions_df = pd.DataFrame({
        'actual': y_test,
        'predicted_prob': y_test_proba,
        'predicted_class_05': (y_test_proba >= 0.5).astype(int),
        'predicted_class_optimal': y_test_pred_optimal
    })
    predictions_df.to_csv('xgb_predictions.csv', index=False)

    # Save feature importance
    feature_imp_df = pd.DataFrame({
        'feature': top_features,
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=False)
    feature_imp_df.to_csv('xgb_feature_importance.csv', index=False)

    # Save model results
    xgb_results_df.to_csv('xgb_model_results.csv', index=False)

    print("\n✅ XGBoost training completed successfully!")
    print(f"🎯 Best Model: {best_xgb_name}")
    print(f"📊 Test Average Precision: {best_xgb_result['Average_Precision']:.4f}")
    print(f"🔒 Holdout Average Precision: {holdout_ap:.4f}")
    print(f"💡 Optimal Threshold: {optimal_threshold:.3f}")
    print(f"🌳 Best Iteration: {best_xgb_result['Best_Iteration']}")

    # 📊 XGBoost Feature Importance Visualization
    plt.figure(figsize=(12, 10))
    top_20_features = feature_imp_df.head(20)
    plt.barh(top_20_features['feature'], top_20_features['importance'])
    plt.title('XGBoost Feature Importance - Top 20 Features')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig('xgb_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("📊 Feature importance plot saved!")

    # 📈 Precision-Recall Curve
    plt.figure(figsize=(10, 8))
    precision, recall, _ = precision_recall_curve(y_test, y_test_proba)
    plt.plot(recall, precision, marker='.')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('XGBoost Precision-Recall Curve')
    plt.grid(True)
    plt.savefig('xgb_precision_recall_curve.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("\n📁 Saved files:")
    print("   - xgb_fraud_model.pkl (Final trained model)")
    print("   - xgb_selected_features.pkl (Feature list)")
    print("   - xgb_holdout_set.csv")
    print("   - xgb_training_set.csv")
    print("   - xgb_predictions.csv")
    print("   - xgb_feature_importance.csv")
    print("   - xgb_model_results.csv")
    print("   - xgb_feature_importance.png")
    print("   - xgb_precision_recall_curve.png")

    # 🎯 Business Impact Analysis
    print("\n" + "="*80)
    print("💼 BUSINESS IMPACT ANALYSIS")
    print("="*80)

    # Calculate cost savings
    fraud_prevented = conf_matrix[1, 1]  # True Positives
    false_positives = conf_matrix[0, 1]  # False Positives

    if len(data[data['is_fraud'] == 1]) > 0:
        avg_fraud_amount = data[data['is_fraud'] == 1]['amt'].mean()
    else:
        avg_fraud_amount = 100  # Fallback average

    false_positive_cost = 10  # Estimated cost per false positive in dollars

    total_savings = fraud_prevented * avg_fraud_amount
    total_fp_cost = false_positives * false_positive_cost
    net_savings = total_savings - total_fp_cost

    print(f"💰 Estimated Fraud Prevented: {fraud_prevented} transactions")
    print(f"💸 Average Fraud Amount: ${avg_fraud_amount:.2f}")
    print(f"📈 Total Fraud Savings: ${total_savings:.2f}")
    print(f"⚠️  False Positives: {false_positives} transactions")
    print(f"🔧 False Positive Cost: ${total_fp_cost:.2f}")
    print(f"🎯 NET SAVINGS: ${net_savings:.2f}")

    # 🎯 Model Inference Function
    def predict_fraud(model, features, transaction_data, threshold=0.5):
        """Helper function for making fraud predictions"""
        # Ensure we have the right features
        missing_features = set(features) - set(transaction_data.columns)
        if missing_features:
            raise ValueError(f"Missing features: {missing_features}")
        
        # Select and order features correctly
        X_pred = transaction_data[features]
        
        # Predict probabilities
        fraud_proba = model.predict_proba(X_pred)[:, 1]
        predictions = (fraud_proba >= threshold).astype(int)
        
        return predictions, fraud_proba

    print(f"\n🎯 Model ready for deployment!")
    print(f"💡 Use threshold: {optimal_threshold:.3f} for optimal performance")
    print(f"📊 Key metric: Average Precision = {holdout_ap:.4f}")

else:
    print("❌ No models were successfully trained!")

print("\n🎯 XGBoost training process completed!")


📈 Training XGBoost variants...
Training XGBoost Scaled Weight...
❌ XGBoost Scaled Weight failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'
Trying fallback for XGBoost Scaled Weight...
✅ XGBoost Scaled Weight (Fallback): AP=0.9112
Training XGBoost SMOTE...
❌ XGBoost SMOTE failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'
Trying fallback for XGBoost SMOTE...
✅ XGBoost SMOTE (Fallback): AP=0.9112
Training XGBoost Balanced...
❌ XGBoost Balanced failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'
Trying fallback for XGBoost Balanced...
✅ XGBoost Balanced (Fallback): AP=0.9270
Training XGBoost High Weight...
❌ XGBoost High Weight failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'
Trying fallback for XGBoost High Weight...
✅ XGBoost High Weight (Fallback): AP=0.9219

🏆 XGBOOST MODEL COMPARISON
                              Model  Average_Precision  